<h2 style="color:#FF7A70;">Clase Proyecto Final: Global Economic Analysis Report 2025</h2>

<p><strong>Curso:</strong> Lenguaje y Programación II</p>

<p><strong>Integrantes del grupo:</strong></p>
<ul>
  <li>Castillo Flores, Hellary Mayte — <em>20240699</em></li>
  <li>Cruz Lozano, Gianella Alejandra — <em>20240705</em></li>
  <li>Tineo Balcázar, Daniela Rosa — <em>20231510</em></li>
</ul>

<h2 style="color:#FF7A70;">Objetivo:</h2>
<p>
Analizar el <strong>Top 10 de países por PIB Nominal</strong> utilizando datos oficiales
del <strong>Banco Mundial</strong>, e incorporar una métrica comparativa adicional llamada
<strong>Capacidad BTC</strong>, basada en el precio actual de Bitcoin, con el fin de
ofrecer una visión económica alternativa y moderna.
</p>

<h2 style="color:#FF7A70;">Fuentes de datos:</h2>
<ul>
  <li><strong>World Bank API</strong> – Indicadores macroeconómicos</li>
  <li><strong>CoinGecko API</strong> – Precio actual de Bitcoin (USD)</li>
</ul>
<h2 style="color:#FF7A70;">Parte 1: Carga de datos y APIs</h2>
<p>En esta sección vamos a:</p>
<ol>
  <li>Descargar el precio de Bitcoin desde CoinGecko.</li>
  <li>Descargar indicadores económicos (PIB, PIB per cápita, crecimiento) desde World Bank.</li>
  <li>Obtener la población de cada país usando REST Countries (optimizado para rapidez).</li>
  <li>Calcular la Capacidad BTC y PIB per cápita real.</li>
</ol>





In [ ]:
#Instalar en caso de no contar con wbgapi y reiniciar el kernel
#%pip install wbgapi requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import requests
import wbgapi as wb
import warnings

warnings.filterwarnings("ignore")#Ignorar advertencias

# -----------------------------
# Transformar valores numéricos en cadenas de texto
# -----------------------------
def format_money(val):
    if pd.isna(val): return "N/A"
    if val >= 1e12: return f"{val/1e12:,.2f} Trillion USD"
    if val >= 1e9: return f"{val/1e9:,.2f} Billion USD"
    return f"{val:,.2f} USD"

def format_btc(val):
    if pd.isna(val): return "N/A"
    return f"{val:,.0f} BTC"

def format_percent(val):
    if pd.isna(val): return "N/A"
    return f"{val:.2f} %"

def format_population(val):
    if pd.isna(val): return "N/A"
    return f"{val:,}"

# -----------------------------
# Cargar datos
# -----------------------------
# Precio BTC
btc_price = requests.get(
    "https://api.coingecko.com/api/v3/simple/price",
    params={"ids": "bitcoin", "vs_currencies": "usd"}
).json()["bitcoin"]["usd"]

# Indicadores económicos
indicadores = {
    "NY.GDP.MKTP.CD": "PIB_Nominal",
    "NY.GDP.PCAP.CD": "PIB_Per_Capita",
    "NY.GDP.MKTP.KD.ZG": "Crecimiento_PIB"
}

df = wb.data.DataFrame(indicadores.keys(), labels=True, mrnev=1).reset_index()
df = df.rename(columns={
    "Country": "Pais",
    "NY.GDP.MKTP.CD": "PIB_Nominal",
    "NY.GDP.PCAP.CD": "PIB_Per_Capita",
    "NY.GDP.MKTP.KD.ZG": "Crecimiento_PIB"
})

# Eliminamos agregados
paises_validos = [c["id"] for c in wb.economy.list() if not c["aggregate"]]
df = df[df["economy"].isin(paises_validos)]
df = df.dropna(subset=["PIB_Nominal"])

# -----------------------------
# Poblaciones (REST Countries) - 1 llamada rápida
# -----------------------------
resp = requests.get("https://restcountries.com/v3.1/all").json()
poblaciones_dict = {
    c["name"]["common"].lower(): c.get("population", None)
    for c in resp
    if isinstance(c, dict) and "name" in c and "common" in c["name"]
}

df["Poblacion"] = df["Pais"].str.lower().map(poblaciones_dict)
df["Poblacion"] = df["Poblacion"].fillna(df["PIB_Nominal"] / df["PIB_Per_Capita"])


# -----------------------------
# Calcular PIB per cápita real y Capacidad BTC
# -----------------------------
df["PIB_Per_Capita_Real"] = df["PIB_Nominal"] / df["Poblacion"]


df["Capacidad_BTC"] = df["PIB_Nominal"] / btc_price

df = df.sort_values("PIB_Nominal", ascending=False).reset_index(drop=True)
df.index += 1

df.head()



ConnectTimeout: HTTPSConnectionPool(host='restcountries.com', port=443): Max retries exceeded with url: /v3.1/all (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001488288D6D0>, 'Connection to restcountries.com timed out. (connect timeout=None)'))

<h2 style="color:#FF7A70;">Parte 2: Menú interactivo y consultas</h2>
<p>Esta sección:</p> 

<ol>
<li>Mostrará el menú de opciones en Jupyter.</li>
<li>Permitirá consultar Top 10 PIB o Capacidad BTC.</li>
<li>Consultará información (completa) de un país específico, al cual accederemos mediante un imput.</li>
</ol>

In [ ]:
# -----------------------------
# Funciones del menú
# -----------------------------
def mostrar_top10(df, columna, titulo):
    print(f"\n🏆 {titulo}")
    print("-" * 50)
    for i, row in df.head(10).iterrows():
        if columna in ["PIB_Nominal", "PIB_Per_Capita", "PIB_Per_Capita_Real"]:
            valor = format_money(row[columna])
        elif columna == "Capacidad_BTC":
            valor = format_btc(row[columna])
        elif columna == "Crecimiento_PIB":
            valor = format_percent(row[columna])
        elif columna == "Poblacion":
            valor = format_population(row[columna])
        else:
            valor = row[columna]
        print(f"{i}. {row['Pais']:<20} {valor}")

def consultar_pais(df):
    pais = input("\nIngrese el país (en inglés): ").strip()
    fila = df[df["Pais"].str.lower() == pais.lower()]
    if fila.empty:
        print("❌ País no encontrado.")
        return None

    row = fila.iloc[0]
    print("\n📊 INFORMACIÓN DEL PAÍS")
    print("-" * 40)
    print(f"País: {row['Pais']}")
    print(f"PIB Nominal: {format_money(row['PIB_Nominal'])}")
    print(f"PIB per Cápita (WB): {format_money(row['PIB_Per_Capita'])}")
    print(f"PIB per Cápita (Real): {format_money(row['PIB_Per_Capita_Real'])}")
    print(f"Crecimiento PIB: {format_percent(row['Crecimiento_PIB'])}")
    print(f"Población: {format_population(row['Poblacion'])}")
    print(f"Capacidad BTC: {format_btc(row['Capacidad_BTC'])}")
    return row

# -----------------------------
# Menú interactivo en Jupyter :D
# -----------------------------
consultas = []

while True:
    print("\n📌 MENÚ PRINCIPAL")
    print("1. Top 10 PIB")
    print("2. Top 10 Capacidad BTC")
    print("3. Consultar país")
    print("4. Salir")
    
    op = input("Seleccione opción: ")
    
    if op == "1":
        mostrar_top10(df, "PIB_Nominal", "TOP 10 PIB")
    elif op == "2":
        mostrar_top10(df.sort_values("Capacidad_BTC", ascending=False), "Capacidad_BTC", "TOP 10 CAPACIDAD BTC")
    elif op == "3":
        r = consultar_pais(df)
        if r is not None:
            consultas.append(r)
    elif op == "4":
        print("✅ Finalizando consultas...")
        break
    else:
        print("❌ Opción inválida")